# Memory

# import

In [1]:
from dotenv import load_dotenv
print(load_dotenv())

from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.messages.human import HumanMessage
from langchain_core.messages.ai import AIMessage

True


# 메모리 없는 경우

In [2]:
llm = ChatOpenAI(temperature=0.1)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI talking to a human"),
    ("human", "{question}")
])

chain = prompt | llm

def invoke_chain(question):
    result = chain.invoke({"question": question})
    print('🐹', result.content)

invoke_chain("My name John")
invoke_chain("What is my name?")

🐹 Hello John! How can I assist you today?
🐹 I'm sorry, but I don't have access to personal information like your name. How can I assist you today?


In [ ]:
# AI 는 요청한 내용에 대한 상태정보를 기억하지 않는다.

# 챗봇은 '대화의 상태'를 기억해야 한다. -> '문맥에 맞는 대화'를 하려면!

# 이전 대화의 내용을 기억하여 AI에 요청해야 한다.

# LangChain 에선 이를 Memory 객체들 제공하여 지원했었다  ~ v0.3.x
#       v1.x.x 부터는 대부분의 Memory 객체들 삭제.  -> LangGraph 를 사용하여 상태정보 유지.

# 메모리 구현 - MessagesPlaceholder

In [6]:
from langchain_core.prompts.chat import MessagesPlaceholder

llm = ChatOpenAI(temperature=0.1)

chat_history = []   # AI 와 주고 받은 Message 들 저장

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI talking to a human"),

    MessagesPlaceholder(variable_name='history'),
    
    ("human", "{question}")
])

chain = prompt | llm

def invoke_chain(question, history):
    result = chain.invoke({"question": question, "history": chat_history})
    print('🐹', result.content)
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=result.content))

print('🔷 chat_history:', chat_history)
invoke_chain("My name John", chat_history)
print('🔷 chat_history:', chat_history)
invoke_chain("What is my name?", chat_history)
print('🔷 chat_history:', chat_history)

🔷 chat_history: []
🐹 Hello John! How can I assist you today?
🔷 chat_history: [HumanMessage(content='My name John', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello John! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
🐹 Your name is John.
🔷 chat_history: [HumanMessage(content='My name John', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello John! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is John.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [7]:
# 매번 chain 호출할때 마다 chat_history 를 넘겨줘야 한다구????
# chain.invoke({"question": question, "history": chat_history})

# 자동으로 chat_hisotry 건네주는 방법 없을까?

# 메모리구현 - RunnablePassthrough

In [12]:
from langchain_core.prompts.chat import MessagesPlaceholder
from langchain_core.runnables.passthrough import RunnablePassthrough

llm = ChatOpenAI(temperature=0.1)

chat_history = []   # AI 와 주고 받은 Message 들 저장

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI talking to a human"),

    MessagesPlaceholder(variable_name='history'),
    
    ("human", "{question}")
])

def load_memory(_):
    return chat_history


# RunnablePassthrough 는 전달받은 key 를 체인의 다음요소까지 전달.
chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm
# chain = {"history":load_memory} | prompt | llm   # question 이 prompt 로 전달 안됨. KeyError

def invoke_chain(question, history):
    result = chain.invoke({"question": question})
    print('🐹', result.content)
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=result.content))

print('🔷 chat_history:', chat_history)
invoke_chain("My name John", chat_history)
print('🔷 chat_history:', chat_history)
invoke_chain("What is my name?", chat_history)
print('🔷 chat_history:', chat_history)    

🔷 chat_history: []
🐹 Hello John! How can I assist you today?
🔷 chat_history: [HumanMessage(content='My name John', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello John! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
🐹 Your name is John.
🔷 chat_history: [HumanMessage(content='My name John', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello John! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is John.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
